In [ ]:
# Reference-sensitivity gate: standalone GPU/JAX Kaggle notebook.
# This single cell checks whether production strong-error and compact Fig. 3
# fitted orders are stable under HH/reference-grid refinement.
# It does not clone, write, or import the local repository.

from __future__ import annotations

import csv
import os
import time
from pathlib import Path

os.environ.setdefault("XLA_PYTHON_CLIENT_MEM_FRACTION", "0.85")
os.environ.setdefault("JAX_ENABLE_X64", "true")

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import jax
import jax.numpy as jnp

jax.config.update("jax_enable_x64", True)

RUN_MODE = os.environ.get("REFERENCE_SENSITIVITY_RUN_MODE", "full").strip().lower()
if RUN_MODE not in {"full", "smoke"}:
    raise ValueError("REFERENCE_SENSITIVITY_RUN_MODE must be 'full' or 'smoke'")

WORKING_ROOT = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd()
OUT_DIR = WORKING_ROOT / "cir_reference_sensitivity_outputs"
FIG_DIR = OUT_DIR / "figures"
RES_DIR = OUT_DIR / "results"
FIG_DIR.mkdir(parents=True, exist_ok=True)
RES_DIR.mkdir(parents=True, exist_ok=True)

MASTER_SEED = 120339106
FINAL_TIME = 1.0
SHARED = dict(kappa=2.0, theta=0.02, x0=0.02)
REGIMES = {
    "A": {"sigma": 0.10},
    "B": {"sigma": 0.20},
    "C": {"sigma": 0.282842712474619},
    "D": {"sigma": 0.50},
    "E": {"sigma": 0.80},
}


def parse_int_list(value: str) -> list[int]:
    return [int(part.strip()) for part in value.split(",") if part.strip()]


def parse_float_list(value: str) -> list[float]:
    return [float(part.strip()) for part in value.split(",") if part.strip()]


def parse_str_list(value: str) -> list[str]:
    return [part.strip() for part in value.split(",") if part.strip()]


if RUN_MODE == "smoke":
    STRONG_N_PATHS = int(os.environ.get("REFSENS_STRONG_N_PATHS", "64"))
    STRONG_REFERENCE_N_STEPS = parse_int_list(
        os.environ.get("REFSENS_STRONG_REFERENCE_N_STEPS", "512,1024")
    )
    STRONG_COARSE_N_STEPS = [8, 16, 32]
    STRONG_REGIME_LIST = ["D", "E"]
    STRONG_SCHEMES = ["KL", "KLM"]
    FIG3_N_PATHS = int(os.environ.get("REFSENS_FIG3_N_PATHS", "16"))
    FIG3_REFERENCE_POWERS = parse_int_list(
        os.environ.get("REFSENS_FIG3_REFERENCE_POWERS", "10,12")
    )
    FIG3_N_A_VALUES = 2
    FIG3_KAPPAS = [2.0]
    FIG3_LEVELS = [4, 5]
else:
    STRONG_N_PATHS = int(os.environ.get("REFSENS_STRONG_N_PATHS", "2000"))
    STRONG_REFERENCE_N_STEPS = parse_int_list(
        os.environ.get("REFSENS_STRONG_REFERENCE_N_STEPS", "4096,16384,32768")
    )
    STRONG_COARSE_N_STEPS = parse_int_list(
        os.environ.get("REFSENS_STRONG_COARSE_N_STEPS", "8,16,32,64,128,256")
    )
    STRONG_REGIME_LIST = parse_str_list(
        os.environ.get("REFSENS_STRONG_REGIMES", "A,C,D,E")
    )
    STRONG_SCHEMES = parse_str_list(
        os.environ.get("REFSENS_STRONG_SCHEMES", "FTE,ProjEuler,KL,KLM")
    )
    FIG3_N_PATHS = int(os.environ.get("REFSENS_FIG3_N_PATHS", "256"))
    FIG3_REFERENCE_POWERS = parse_int_list(
        os.environ.get("REFSENS_FIG3_REFERENCE_POWERS", "14,16,18")
    )
    FIG3_N_A_VALUES = int(os.environ.get("REFSENS_FIG3_N_A_VALUES", "8"))
    FIG3_KAPPAS = parse_float_list(os.environ.get("REFSENS_FIG3_KAPPAS", "2.0,0.2"))
    FIG3_LEVELS = parse_int_list(os.environ.get("REFSENS_FIG3_LEVELS", "4,5,6,7,8,9"))

RHO = float(os.environ.get("REFSENS_RHO", "64"))
FIG3_LAMBDA = 0.05
FIG3_INITIAL_X = 0.02


def write_csv(path: Path, rows: list[dict]) -> None:
    if not rows:
        return
    with path.open("w", newline="", encoding="utf-8") as file:
        writer = csv.DictWriter(file, fieldnames=list(rows[0].keys()))
        writer.writeheader()
        writer.writerows(rows)


def fit_loglog_order(step_sizes, errors):
    h = np.asarray(step_sizes, dtype=float)
    e = np.maximum(np.asarray(errors, dtype=float), 1e-300)
    return float(np.polyfit(np.log(h), np.log(e), 1)[0])


def cir_delta(kappa, theta, sigma):
    return 4.0 * kappa * theta / sigma**2


def kl_alpha(kappa, theta, sigma):
    return (4.0 * kappa * theta - sigma**2) / 8.0


def aggregate_dW(dW_fine, factor):
    n_paths, n_fine = dW_fine.shape
    return dW_fine.reshape((n_paths, n_fine // factor, factor)).sum(axis=2)


def hh_step(x, kappa, theta, sigma, dt, dW):
    floor = 0.25 * sigma**2 * dt
    r1 = jnp.maximum(
        0.5 * sigma * jnp.sqrt(dt),
        jnp.sqrt(jnp.maximum(floor, x)) + 0.5 * sigma * dW,
    )
    x_hat = r1 * r1 + dt * (kappa * (theta - x) - 0.25 * sigma**2)
    return jnp.maximum(x_hat, 0.0)


def hh_terminal_from_dW_jax(X0, kappa, theta, sigma, dt, dW):
    def step(x, dW_col):
        return hh_step(x, kappa, theta, sigma, dt, dW_col), None
    x0 = jnp.full((dW.shape[0],), X0, dtype=jnp.float64)
    x, _ = jax.lax.scan(step, x0, dW.T)
    return x


def fte_terminal_from_dW_jax(X0, kappa, theta, sigma, dt, dW):
    def step(x_aux, dW_col):
        x_pos = jnp.maximum(x_aux, 0.0)
        x_next = x_aux + kappa * (theta - x_pos) * dt + sigma * jnp.sqrt(x_pos) * dW_col
        return x_next, None
    x0 = jnp.full((dW.shape[0],), X0, dtype=jnp.float64)
    x_aux, _ = jax.lax.scan(step, x0, dW.T)
    return jnp.maximum(x_aux, 0.0)


def projected_terminal_from_dW_jax(X0, kappa, theta, sigma, dt, dW):
    alpha = kl_alpha(kappa, theta, sigma)
    y_floor = 0.5 * sigma * dt ** 0.5
    def step(y, dW_col):
        y_safe = jnp.maximum(y, y_floor)
        y_hat = y_safe + (alpha / y_safe - 0.5 * kappa * y_safe) * dt + 0.5 * sigma * dW_col
        return jnp.maximum(y_hat, y_floor), None
    y0 = jnp.full((dW.shape[0],), max(np.sqrt(X0), y_floor), dtype=jnp.float64)
    y, _ = jax.lax.scan(step, y0, dW.T)
    return y * y


def kl_uniform_terminal_from_dW_jax(X0, kappa, theta, sigma, dt, dW):
    alpha = kl_alpha(kappa, theta, sigma)
    decay = np.exp(-kappa * dt)
    def step(x, dW_col):
        return decay * (jnp.sqrt(x + 2.0 * alpha * dt) + 0.5 * sigma * dW_col) ** 2, None
    x0 = jnp.full((dW.shape[0],), X0, dtype=jnp.float64)
    x, _ = jax.lax.scan(step, x0, dW.T)
    return x


def soft_zero_threshold(kappa, theta, dt_max, rho=2.0):
    return theta * (1.0 - np.exp(-kappa * dt_max)) / rho


def kl_adaptive_terminal_from_fine_dW_jax(X0, kappa, theta, sigma, T, dt_max, dW_fine, rho=2.0, safety=0.95):
    alpha = kl_alpha(kappa, theta, sigma)
    X_zero = soft_zero_threshold(kappa, theta, dt_max, rho)
    dW_fine = jnp.asarray(dW_fine, dtype=jnp.float64)
    n_paths, n_fine = dW_fine.shape
    dt_fine = T / n_fine
    W = jnp.concatenate([jnp.zeros((n_paths, 1), dtype=jnp.float64), jnp.cumsum(dW_fine, axis=1)], axis=1)
    rows = jnp.arange(n_paths)
    x0 = jnp.full((n_paths,), X0, dtype=jnp.float64)
    pos0 = jnp.zeros((n_paths,), dtype=jnp.int64)
    counters0 = jnp.zeros(2, dtype=jnp.int64)

    def cond(state):
        _, pos, _ = state
        return jnp.any(pos < n_fine)

    def body(state):
        x, pos, counters = state
        active = pos < n_fine
        remaining_steps = n_fine - pos
        remaining_time = remaining_steps * dt_fine
        in_soft_zero = active & (x < X_zero)
        in_splitting = active & (~in_soft_zero)

        x_for_soft = jnp.minimum(x, X_zero * 0.999999)
        dt_sz = -jnp.log((X_zero - theta) / (x_for_soft - theta)) / kappa
        if alpha < 0.0:
            dt_adaptive = safety * x / (2.0 * abs(alpha))
            split_h = jnp.minimum(jnp.minimum(dt_adaptive, dt_max), remaining_time)
        else:
            split_h = jnp.minimum(dt_max, remaining_time)

        h_cont = jnp.where(in_soft_zero, jnp.minimum(dt_sz, remaining_time), 0.0)
        h_cont = jnp.where(in_splitting, split_h, h_cont)
        m = jnp.floor(h_cont / dt_fine).astype(jnp.int64)
        m = jnp.where(active, jnp.maximum(m, 1), 0)
        m = jnp.minimum(m, remaining_steps)
        h = m * dt_fine
        dW = W[rows, pos + m] - W[rows, pos]

        decay = jnp.exp(-kappa * h)
        x_soft = decay * x + theta * (1.0 - decay)
        inside_sqrt = jnp.maximum(x + 2.0 * alpha * h, 0.0)
        x_split = decay * (jnp.sqrt(inside_sqrt) + 0.5 * sigma * dW) ** 2
        x_next = jnp.where(in_soft_zero, x_soft, jnp.where(in_splitting, x_split, x))

        counters = counters + jnp.array(
            [jnp.sum(active), jnp.sum(in_soft_zero)],
            dtype=jnp.int64,
        )
        return x_next, pos + m, counters

    x, _, counters = jax.lax.while_loop(cond, body, (x0, pos0, counters0))
    total = int(counters[0])
    return x, {
        "n_steps_total": total,
        "mean_steps_per_path": total / n_paths,
        "soft_zero_fraction": int(counters[1]) / max(total, 1),
    }


def implicit_step(y, h, dW, alpha, beta, gamma):
    a = 1.0 - beta * h
    u = y + gamma * dW
    return (u + jnp.sqrt(u * u + 4.0 * a * alpha * h)) / (2.0 * a)


def projected_backstop_step(y, h, dW, alpha, beta, gamma):
    y_floor = gamma * jnp.sqrt(h)
    y_hat = y + (alpha / y + beta * y) * h + gamma * dW
    return jnp.maximum(y_hat, y_floor)


def klm_terminal_from_fine_dW_jax(X0, kappa, theta, sigma, T, h_max, dW_fine, rho=64.0):
    alpha = kl_alpha(kappa, theta, sigma)
    beta = -0.5 * kappa
    gamma = 0.5 * sigma
    use_implicit = alpha > 0.0
    n_paths, n_fine = dW_fine.shape
    dt_fine = T / n_fine
    h_min = h_max / rho
    m_min = max(int(round(h_min / dt_fine)), 1)
    m_max = max(int(np.floor(h_max / dt_fine)), 1)
    W = jnp.concatenate([jnp.zeros((n_paths, 1), dtype=jnp.float64), jnp.cumsum(dW_fine, axis=1)], axis=1)
    rows = jnp.arange(n_paths)
    y0 = jnp.full((n_paths,), jnp.sqrt(X0), dtype=jnp.float64)
    pos0 = jnp.zeros((n_paths,), dtype=jnp.int64)
    counters0 = jnp.zeros(3, dtype=jnp.int64)
    def cond(state):
        _, pos, _ = state
        return jnp.any(pos < n_fine)
    def body(state):
        y, pos, counters = state
        active = pos < n_fine
        h_prop = h_max * jnp.minimum(1.0, jnp.abs(y))
        m = jnp.floor(h_prop / dt_fine).astype(jnp.int64)
        min_triggered = m < m_min
        m = jnp.where(min_triggered, m_min, jnp.minimum(m, m_max))
        m = jnp.minimum(m, n_fine - pos)
        h = m * dt_fine
        dW = W[rows, pos + m] - W[rows, pos]
        y_explicit = jnp.where(h > 0.0, y + (alpha / y + beta * y) * h + gamma * dW, y)
        y_imp = implicit_step(y, h, dW, alpha, beta, gamma)
        y_proj = projected_backstop_step(y, h, dW, alpha, beta, gamma)
        y_backstop = jnp.where(use_implicit, y_imp, y_proj)
        neg_retake = (~min_triggered) & (y_explicit <= 0.0)
        use_backstop = min_triggered | neg_retake
        y_next = jnp.where(use_backstop, y_backstop, y_explicit)
        counters = counters + jnp.array(
            [jnp.sum(active), jnp.sum(active & min_triggered), jnp.sum(active & neg_retake)],
            dtype=jnp.int64,
        )
        return y_next, pos + m, counters
    y, _, counters = jax.lax.while_loop(cond, body, (y0, pos0, counters0))
    total = int(counters[0])
    return y * y, {
        "n_steps_total": total,
        "n_backstop_min": int(counters[1]),
        "n_backstop_neg": int(counters[2]),
        "backstop_fraction": (int(counters[1]) + int(counters[2])) / max(total, 1),
    }


def if_terminal_from_fine_dW_jax(X0, kappa, theta, sigma, T, dW_fine):
    alpha = kl_alpha(kappa, theta, sigma)
    beta = -0.5 * kappa
    gamma = 0.5 * sigma
    dW_fine = jnp.asarray(dW_fine, dtype=jnp.float64)
    n_paths, n_fine = dW_fine.shape
    dt = T / n_fine

    def step(y, dW_col):
        return implicit_step(y, dt, dW_col, alpha, beta, gamma), None

    y0 = jnp.full((n_paths,), jnp.sqrt(X0), dtype=jnp.float64)
    y, _ = jax.lax.scan(step, y0, dW_fine.T)
    return y * y


def sigma_from_a(a, kappa, lam):
    return float(np.sqrt(2.0 * kappa * lam * a))


def require_nested(reference_values: list[int], coarse_values: list[int] | None = None) -> None:
    max_ref = max(reference_values)
    for ref in reference_values:
        if max_ref % ref != 0:
            raise ValueError(f"max reference {max_ref} must divide reference {ref}")
        if coarse_values is not None:
            for coarse in coarse_values:
                if ref % coarse != 0:
                    raise ValueError(f"reference {ref} must divide coarse level {coarse}")


def fitted_orders_for_metric(rows, x_key, y_key, group_keys):
    orders = []
    groups = sorted({tuple(row[key] for key in group_keys) for row in rows})
    for group in groups:
        selected = [
            row for row in rows
            if tuple(row[key] for key in group_keys) == group
        ]
        if len(selected) < 2:
            continue
        order = fit_loglog_order(
            [row[x_key] for row in selected],
            [row[y_key] for row in selected],
        )
        out = {key: value for key, value in zip(group_keys, group, strict=True)}
        out["fitted_order"] = order
        orders.append(out)
    return orders


def run_one_strong_scheme(scheme, params, dt, dW, dW_ref):
    if scheme == "FTE":
        terminal = fte_terminal_from_dW_jax(
            params["x0"], params["kappa"], params["theta"], params["sigma"], dt, dW
        )
        return terminal, "fixed", float(dW.shape[1]), 0.0
    if scheme == "ProjEuler":
        terminal = projected_terminal_from_dW_jax(
            params["x0"], params["kappa"], params["theta"], params["sigma"], dt, dW
        )
        return terminal, "fixed", float(dW.shape[1]), 0.0
    if scheme == "KL":
        if kl_alpha(params["kappa"], params["theta"], params["sigma"]) < 0.0:
            terminal, stats = kl_adaptive_terminal_from_fine_dW_jax(
                params["x0"], params["kappa"], params["theta"], params["sigma"],
                FINAL_TIME, dt, dW_ref,
            )
            return (
                terminal,
                "adaptive-soft-zero",
                float(stats["mean_steps_per_path"]),
                float(stats["soft_zero_fraction"]),
            )
        terminal = kl_uniform_terminal_from_dW_jax(
            params["x0"], params["kappa"], params["theta"], params["sigma"], dt, dW
        )
        return terminal, "uniform", float(dW.shape[1]), 0.0
    if scheme == "KLM":
        terminal, stats = klm_terminal_from_fine_dW_jax(
            params["x0"], params["kappa"], params["theta"], params["sigma"],
            FINAL_TIME, dt, dW_ref, rho=RHO,
        )
        return (
            terminal,
            "adaptive-backstop",
            float(stats["n_steps_total"] / STRONG_N_PATHS),
            float(stats["backstop_fraction"]),
        )
    raise ValueError(f"unknown scheme: {scheme}")


def run_strong_reference_sensitivity():
    require_nested(STRONG_REFERENCE_N_STEPS, STRONG_COARSE_N_STEPS)
    rows = []
    max_ref = max(STRONG_REFERENCE_N_STEPS)
    dt_max = FINAL_TIME / max_ref

    for regime_index, regime_name in enumerate(STRONG_REGIME_LIST):
        sigma = REGIMES[regime_name]["sigma"]
        params = dict(**SHARED, sigma=sigma)
        key = jax.random.fold_in(jax.random.PRNGKey(MASTER_SEED), regime_index)
        dW_max = jax.random.normal(
            key, (STRONG_N_PATHS, max_ref), dtype=jnp.float64
        ) * jnp.sqrt(dt_max)

        for reference_n_steps in STRONG_REFERENCE_N_STEPS:
            ref_factor = max_ref // reference_n_steps
            dW_ref = dW_max if ref_factor == 1 else aggregate_dW(dW_max, ref_factor)
            dt_ref = FINAL_TIME / reference_n_steps
            reference = hh_terminal_from_dW_jax(
                params["x0"], params["kappa"], params["theta"], sigma, dt_ref, dW_ref
            )
            reference.block_until_ready()

            for scheme in STRONG_SCHEMES:
                for n_steps in STRONG_COARSE_N_STEPS:
                    factor = reference_n_steps // n_steps
                    dW = aggregate_dW(dW_ref, factor)
                    dt = FINAL_TIME / n_steps
                    start = time.perf_counter()
                    terminal, variant, mean_steps, backstop_fraction = run_one_strong_scheme(
                        scheme, params, dt, dW, dW_ref
                    )
                    terminal.block_until_ready()
                    runtime = time.perf_counter() - start
                    diff = terminal - reference
                    rows.append({
                        "sensitivity_kind": "strong_error",
                        "regime": regime_name,
                        "scheme": scheme,
                        "scheme_variant": variant,
                        "reference": "HH",
                        "reference_n_steps": reference_n_steps,
                        "dt": dt,
                        "l1": float(jnp.mean(jnp.abs(diff))),
                        "l2": float(jnp.sqrt(jnp.mean(diff * diff))),
                        "runtime_s": runtime,
                        "mean_steps_per_path": mean_steps,
                        "backstop_fraction": backstop_fraction,
                    })

    orders = fitted_orders_for_metric(
        rows,
        x_key="dt",
        y_key="l2",
        group_keys=["regime", "scheme", "reference_n_steps"],
    )
    for row in orders:
        row["sensitivity_kind"] = "strong_error"

    write_csv(RES_DIR / "strong_reference_sensitivity.csv", rows)
    write_csv(RES_DIR / "strong_reference_sensitivity_orders.csv", orders)
    return rows, orders


def run_fig3_reference_sensitivity():
    powers_as_steps = [2**power for power in FIG3_REFERENCE_POWERS]
    require_nested(powers_as_steps)
    rows = []
    max_power = max(FIG3_REFERENCE_POWERS)
    max_steps = 2**max_power
    dt_max = FINAL_TIME / max_steps
    a_values = np.concatenate(([0.0], np.linspace(0.04, 1.6, FIG3_N_A_VALUES)))

    for kappa_index, kappa in enumerate(FIG3_KAPPAS):
        key = jax.random.fold_in(jax.random.PRNGKey(MASTER_SEED), 100 + kappa_index)
        dW_max = jax.random.normal(
            key, (FIG3_N_PATHS, max_steps), dtype=jnp.float64
        ) * jnp.sqrt(dt_max)

        for reference_power in FIG3_REFERENCE_POWERS:
            n_fine = 2**reference_power
            ref_factor = max_steps // n_fine
            dW_ref = dW_max if ref_factor == 1 else aggregate_dW(dW_max, ref_factor)

            for a in a_values:
                sigma = sigma_from_a(a, kappa, FIG3_LAMBDA)
                reference = if_terminal_from_fine_dW_jax(
                    FIG3_INITIAL_X, kappa, FIG3_LAMBDA, sigma, FINAL_TIME, dW_ref
                )
                reference.block_until_ready()

                for level in FIG3_LEVELS:
                    hmax = 2.0 ** (-level)
                    start = time.perf_counter()
                    terminal, stats = klm_terminal_from_fine_dW_jax(
                        FIG3_INITIAL_X, kappa, FIG3_LAMBDA, sigma,
                        FINAL_TIME, hmax, dW_ref, rho=RHO,
                    )
                    terminal.block_until_ready()
                    runtime = time.perf_counter() - start
                    diff = terminal - reference
                    rows.append({
                        "sensitivity_kind": "klm_fig3_jax",
                        "kappa": float(kappa),
                        "a": float(a),
                        "sigma": float(sigma),
                        "level": int(level),
                        "hmax": float(hmax),
                        "reference_power": int(reference_power),
                        "reference_step": float(2.0 ** (-reference_power)),
                        "number_of_fine_steps": int(n_fine),
                        "n_paths": int(FIG3_N_PATHS),
                        "rmse": float(jnp.sqrt(jnp.mean(diff * diff))),
                        "backstop_fraction": float(stats["backstop_fraction"]),
                        "mean_steps_per_path": float(stats["n_steps_total"] / FIG3_N_PATHS),
                        "runtime_s": runtime,
                    })

    orders = fitted_orders_for_metric(
        rows,
        x_key="hmax",
        y_key="rmse",
        group_keys=["kappa", "a", "reference_power"],
    )
    for row in orders:
        row["sensitivity_kind"] = "klm_fig3_jax"

    write_csv(RES_DIR / "fig3_reference_sensitivity.csv", rows)
    write_csv(RES_DIR / "fig3_reference_sensitivity_orders.csv", orders)
    return rows, orders


def plot_strong_order_sensitivity(order_rows):
    if not order_rows:
        return
    fig, ax = plt.subplots(figsize=(7.0, 4.5))
    for regime, scheme in sorted({(r["regime"], r["scheme"]) for r in order_rows}):
        selected = [r for r in order_rows if r["regime"] == regime and r["scheme"] == scheme]
        selected.sort(key=lambda row: row["reference_n_steps"])
        ax.plot(
            [r["reference_n_steps"] for r in selected],
            [r["fitted_order"] for r in selected],
            "o-",
            label=f"{regime} {scheme}",
        )
    ax.set_xscale("log", base=2)
    ax.axhline(0.5, color="k", ls="--", lw=0.8)
    ax.axhline(1.0, color="k", ls=":", lw=0.8)
    ax.set_xlabel("HH reference steps")
    ax.set_ylabel("fitted strong L2 order")
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=7, ncol=2)
    fig.tight_layout()
    fig.savefig(FIG_DIR / "strong_reference_sensitivity_orders.pdf")
    plt.close(fig)


def plot_fig3_order_sensitivity(order_rows):
    if not order_rows:
        return
    fig, ax = plt.subplots(figsize=(7.0, 4.5))
    for kappa, power in sorted({(r["kappa"], r["reference_power"]) for r in order_rows}):
        selected = [
            r for r in order_rows
            if r["kappa"] == kappa and r["reference_power"] == power and r["a"] > 0.0
        ]
        selected.sort(key=lambda row: row["a"])
        ax.plot(
            [r["a"] for r in selected],
            [r["fitted_order"] for r in selected],
            "o-",
            label=f"kappa={kappa:g}, ref=2^-{int(power)}",
        )
    ax.axhline(0.5, color="k", ls="--", lw=0.8)
    ax.axhline(1.0, color="k", ls=":", lw=0.8)
    ax.set_xlabel("a = sigma^2 / (2 kappa lambda)")
    ax.set_ylabel("fitted strong L2 order")
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=7, ncol=2)
    fig.tight_layout()
    fig.savefig(FIG_DIR / "fig3_reference_sensitivity_orders.pdf")
    plt.close(fig)


def archive_outputs() -> None:
    import shutil

    archive = shutil.make_archive(str(OUT_DIR), "zip", OUT_DIR)
    print(f"Archived outputs to {archive}")


def main() -> None:
    print(f"Standalone reference-sensitivity gate, RUN_MODE={RUN_MODE}")
    print("JAX devices:", jax.devices())
    print(f"Output directory: {OUT_DIR}")
    print("Strong reference steps:", STRONG_REFERENCE_N_STEPS)
    print("Fig. 3 reference powers:", FIG3_REFERENCE_POWERS)
    strong_rows, strong_orders = run_strong_reference_sensitivity()
    print(f"Strong rows={len(strong_rows)}, order rows={len(strong_orders)}")
    fig3_rows, fig3_orders = run_fig3_reference_sensitivity()
    print(f"Fig. 3 rows={len(fig3_rows)}, order rows={len(fig3_orders)}")
    plot_strong_order_sensitivity(strong_orders)
    plot_fig3_order_sensitivity(fig3_orders)
    archive_outputs()
    print("Done.")


main()
